# Retinal Vessel Segmentation

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Technologies Used:**
* **Language:** Python
* **Libraries:** TODO

## Imports

In [1]:
#General imports
import os
import cv2
from skimage import io
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from imblearn.metrics import geometric_mean_score, specificity_score, sensitivity_score

#Imports for Retinal Vessel Detection via Image Processing
from skimage.filters import threshold_otsu, gaussian, sobel
from skimage.morphology import opening, closing
from skimage.color import rgb2gray
from skimage import exposure

#Imports for Retinal Vessel Detection via Traditional Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score
from numpy.lib.stride_tricks import sliding_window_view

#Imports for Retinal Vessel Detection via Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from skimage.transform import resize


## Images
From HRF image database: https://www5.cs.fau.de/research/data/fundus-images/

In [2]:
image_names = [f"{str(i).zfill(2)}_{suffix}" for i in range(1, 16) for suffix in ["h", "g", "dr"]]

def load_images(image_name):
    raw_image = io.imread(f"../data/images/{image_name}.jpg")
    gs_image = io.imread(f"../data/goldstandard/{image_name}.tif")
    mask = io.imread(f"../data/fovs/{image_name}_mask.tif")
    mask = mask[..., 0]
    return raw_image, gs_image, mask


## Quality metrics and visualisation

In [3]:
def generate_overlay(raw_image, mask_image):
    pred = mask_image > 0

    overlay = raw_image.copy()
    overlay[pred] = [0, 255, 0]

    return overlay

def calculate_metrics(gs_image, generated_image, mask = None):
    valid = mask > 0

    y_true = (gs_image > 0)[valid].astype(int)
    y_generated = (generated_image > 0)[valid].astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_generated, labels=[0, 1]).ravel()

    accuracy = accuracy_score(y_true, y_generated)

    sensitivity = sensitivity_score(y_true, y_generated)

    specificity = specificity_score(y_true, y_generated)

    gmean = geometric_mean_score(y_true, y_generated, average='binary')

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "G-Mean": gmean
    }

def print_average_metrics(metrics_per_image):
    mean_accuracy = np.mean([m["Accuracy"] for _, m in metrics_per_image])
    mean_sensitivity = np.mean([m["Sensitivity"] for _, m in metrics_per_image])
    mean_specificity = np.mean([m["Specificity"] for _, m in metrics_per_image])
    mean_gmean = np.mean([m["G-Mean"] for _, m in metrics_per_image])

    total_tp = sum(m["TP"] for _, m in metrics_per_image)
    total_tn = sum(m["TN"] for _, m in metrics_per_image)
    total_fp = sum(m["FP"] for _, m in metrics_per_image)
    total_fn = sum(m["FN"] for _, m in metrics_per_image)

    print("Średnie metryki na obrazach testowych:")
    print()
    print(f"{'Accuracy':12s}: {mean_accuracy:.4f}")
    print(f"{'Sensitivity':12s}: {mean_sensitivity:.4f}")
    print(f"{'Specificity':12s}: {mean_specificity:.4f}")
    print(f"{'G-Mean':12s}: {mean_gmean:.4f}")
    print()
    print("Suma wartości macierzy pomyłek:")
    print(f"{'TP':12s}: {total_tp}")
    print(f"{'TN':12s}: {total_tn}")
    print(f"{'FP':12s}: {total_fp}")
    print(f"{'FN':12s}: {total_fn}")

## Retinal Vessel Detection Functions

### Retinal Vessel Detection via Image processing

In [4]:
def retinal_vessel_segmentation_image_processing(image, mask):
    ######## Pre-processing ########
    image = rgb2gray(image)

    image = gaussian(image, sigma=1)

    image = exposure.equalize_hist(image)

    ######## Core processing ########
    image = sobel(image)

    ######## FOV only ########
    valid = mask > 0

    image_fov = image[valid]

    threshold = threshold_otsu(image_fov)

    ######## Post-processing ########
    generated_image = image > threshold

    generated_image[~valid] = False

    generated_image = closing(generated_image)
    generated_image = opening(generated_image)

    generated_image[~valid] = False

    return generated_image

### Retinal Vessel Detection via Traditional Machine Learning

In [5]:
def retinal_vessel_segmentation_machine_learning(image, mask, classifier, patch_size=5, batch_size=50000):
    green = image[:, :, 1].astype(np.float32)

    if green.max() > 1:
        green = green / 255.0

    green = exposure.equalize_adapthist(
        green,
        clip_limit=0.03
    ).astype(np.float32)

    h, w = green.shape
    half = patch_size // 2

    if mask is None:
        valid = np.ones((h, w), dtype=bool)
    else:
        valid = mask > 0

    valid = valid.copy()

    ############ Cut out image edges #################
    valid[:half, :] = False
    valid[-half:, :] = False
    valid[:, :half] = False
    valid[:, -half:] = False

    ys, xs = np.where(valid)

    result = np.zeros((h, w), dtype=np.uint8)

    windows = sliding_window_view(green, (patch_size, patch_size))

    ############ Predict pixels in batches #################
    for start in range(0, len(ys), batch_size):
        end = start + batch_size

        yy = ys[start:end]
        xx = xs[start:end]

        patches = windows[
            yy - half,
            xx - half
        ]

        ############ Extract patch features #################
        X = np.column_stack([
            np.mean(patches, axis=(1, 2)),
            np.var(patches, axis=(1, 2)),
            np.min(patches, axis=(1, 2)),
            np.max(patches, axis=(1, 2)),
            patches[:, half, half],
            np.std(patches, axis=(1, 2))
        ]).astype(np.float32)

        pred = classifier.predict(X)

        result[yy, xx] = pred.astype(np.uint8)

    if mask is not None:
        result[mask <= 0] = 0

    return result


def extract_patch_features(patch):
    half = patch.shape[0] // 2

    features = [
        np.mean(patch),
        np.var(patch),
        np.min(patch),
        np.max(patch),
        patch[half, half],
        np.std(patch)
    ]

    return features


def extract_features_from_image(image, gs_image, mask, samples_per_image=4000, patch_size=5, random_state=42):
    ############ Pre-processing #################
    green = image[:, :, 1]
    green = green.astype(np.float32)

    if green.max() > 1:
        green = green / 255.0

    green = exposure.equalize_adapthist(
        green,
        clip_limit=0.03
    )

    ############ Patch extracting #################
    rng = np.random.default_rng(random_state)
    h, w = green.shape
    half = patch_size // 2

    expert = gs_image > 0

    valid = mask > 0

    ############ Cut out edges #################
    valid[:half, :] = False
    valid[-half:, :] = False
    valid[:, :half] = False
    valid[:, -half:] = False

    vessel_pixels = valid & expert
    background_pixels = valid & ~expert

    v_y, v_x = np.where(vessel_pixels)
    b_y, b_x = np.where(background_pixels)

    ############ Undersampling 50:50 #################
    n_samples = min(
        samples_per_image // 2,
        len(v_y),
        len(b_y)
    )

    idx_v = rng.choice(len(v_y), n_samples, replace=False)
    idx_b = rng.choice(len(b_y), n_samples, replace=False)

    ############ Create labels #################
    selected_y = np.concatenate([
        v_y[idx_v],
        b_y[idx_b]
    ])

    selected_x = np.concatenate([
        v_x[idx_v],
        b_x[idx_b]
    ])

    labels = np.concatenate([
        np.ones(n_samples),
        np.zeros(n_samples)
    ])

    ############ Extract features for each pixel #################
    features = []

    for y, x in zip(selected_y, selected_x):
        patch = green[
            y - half : y + half + 1,
            x - half : x + half + 1
        ]

        features.append(extract_patch_features(patch))

    X = np.array(features, dtype=np.float32)
    y = labels.astype(int)

    ############ Shuffling #################
    permutation = rng.permutation(len(y))

    X = X[permutation]
    y = y[permutation]

    return X, y


def build_dataset_from_images(selected_image_names, samples_per_image=4000, patch_size=5, random_state=42):
    x_all = []
    y_all = []

    for i, image_name in enumerate(selected_image_names):
        print(f"Przetwarzanie obrazu: {image_name}")
        raw_image, gs_image, mask = load_images(image_name)

        X_img, y_img = extract_features_from_image(
            raw_image,
            gs_image,
            mask,
            samples_per_image=samples_per_image,
            patch_size=patch_size,
            random_state=random_state + i
        )

        x_all.append(X_img)
        y_all.append(y_img)

    x_all = np.vstack(x_all)
    y_all = np.concatenate(y_all)

    return x_all, y_all


def train_random_forest_retinal_classifier(train_image_names, samples_per_image=4000, patch_size=5, random_state=42):
    ############ Dataset from train images #################
    x_train, y_train = build_dataset_from_images(
        selected_image_names=train_image_names,
        samples_per_image=samples_per_image,
        patch_size=patch_size,
        random_state=random_state
    )


    ############ Classifier #################
    classifier = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=random_state,
        n_jobs=-1
    )

    ############ Classifier training #################
    classifier.fit(x_train, y_train)

    print("Wytrenowano klasyfikator.")

    return classifier

### Retinal Vessel Detection via Deep Learning

In [6]:
UNET_MODEL_PATH = "unetModel.keras"
PATCH_SIZE_UNET = 128


def _preprocess_green_channel_clahe(img_rgb):
    """CLAHE na kanale zielonym tak jak w prostym patch-based U-Net."""
    green = img_rgb[:, :, 1]

    if green.dtype != np.uint8:
        green = green.astype(np.float32)
        if green.max() <= 1.0:
            green = green * 255.0
        green = np.clip(green, 0, 255).astype(np.uint8)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    green = clahe.apply(green)
    green = green.astype(np.float32) / 255.0

    return green


def build_simple_unet(input_shape=(128, 128, 1)):
    inputs = layers.Input(input_shape)

    # Encoder
    c1 = layers.Conv2D(16, (3, 3), padding="same")(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation("relu")(c1)
    c1 = layers.Conv2D(16, (3, 3), padding="same")(c1)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation("relu")(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(32, (3, 3), padding="same")(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation("relu")(c2)
    c2 = layers.Conv2D(32, (3, 3), padding="same")(c2)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation("relu")(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Bottleneck
    c3 = layers.Conv2D(64, (3, 3), padding="same")(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation("relu")(c3)
    c3 = layers.Conv2D(64, (3, 3), padding="same")(c3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation("relu")(c3)

    # Decoder
    u4 = layers.Conv2DTranspose(
        32,
        (2, 2),
        strides=(2, 2),
        padding="same"
    )(c3)
    u4 = layers.concatenate([u4, c2])
    c4 = layers.Conv2D(32, (3, 3), padding="same")(u4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation("relu")(c4)

    u5 = layers.Conv2DTranspose(
        16,
        (2, 2),
        strides=(2, 2),
        padding="same"
    )(c4)
    u5 = layers.concatenate([u5, c1])
    c5 = layers.Conv2D(16, (3, 3), padding="same")(u5)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Activation("relu")(c5)

    outputs = layers.Conv2D(1, (1, 1), activation="sigmoid")(c5)

    model = models.Model(
        inputs=[inputs],
        outputs=[outputs]
    )

    return model


# Alias zostawiony dla kompatybilności ze starszymi komórkami notebooka.
def build_unet(input_shape=(128, 128, 1), base_filters=16):
    return build_simple_unet(input_shape=input_shape)


def prepare_dl_data_patches(
    selected_image_names,
    patch_size=128,
    patches_per_img=150
):
    X_train_dl = []
    y_train_dl = []

    for image_name in selected_image_names:
        print(f"Przetwarzanie obrazu: {image_name}")

        raw_image, expert_mask, fov_mask = load_images(image_name)

        green = _preprocess_green_channel_clahe(raw_image)

        mask = (expert_mask > 0).astype(np.float32)
        fov = fov_mask > 0

        green[~fov] = 0
        mask[~fov] = 0

        h, w = green.shape

        if h < patch_size or w < patch_size:
            raise ValueError(
                f"Obraz {image_name} ma rozmiar {h}x{w}, "
                f"mniejszy niż patch_size={patch_size}."
            )

        for _ in range(patches_per_img):
            y = np.random.randint(0, h - patch_size + 1)
            x = np.random.randint(0, w - patch_size + 1)

            patch_img = green[
                y:y + patch_size,
                x:x + patch_size
            ]

            patch_mask = mask[
                y:y + patch_size,
                x:x + patch_size
            ]

            X_train_dl.append(patch_img)
            y_train_dl.append(patch_mask)

    X_train_dl = np.array(X_train_dl, dtype=np.float32)
    X_train_dl = np.expand_dims(X_train_dl, axis=-1)

    y_train_dl = np.array(y_train_dl, dtype=np.float32)
    y_train_dl = np.expand_dims(y_train_dl, axis=-1)

    return X_train_dl, y_train_dl


@tf.keras.utils.register_keras_serializable()
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)

    intersection = K.sum(y_true_f * y_pred_f)

    return (2.0 * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )


@tf.keras.utils.register_keras_serializable()
def dice_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - dice_coef(
        y_true,
        y_pred,
        smooth=smooth
    )


@tf.keras.utils.register_keras_serializable()
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)

    return bce + dice


def train_unet(
    selected_image_names=None,
    num_train=15,
    patch_size=128,
    patches_per_img=150,
    epochs=40,
    batch_size=16,
    learning_rate=0.001,
    model_path=UNET_MODEL_PATH,
    random_state=42
):
    global PATCH_SIZE_UNET
    global unet_history

    tf.keras.utils.set_random_seed(random_state)
    np.random.seed(random_state)

    if selected_image_names is None:
        selected_image_names = image_names[:num_train]
    else:
        selected_image_names = list(selected_image_names)[:num_train]

    PATCH_SIZE_UNET = patch_size

    print("Przygotowywanie danych dla TensorFlow U-Net (Patch-Based)...")
    print(f"Obrazy treningowe: {len(selected_image_names)}")
    print(f"Patche na obraz: {patches_per_img}")
    print(f"Patch size: {patch_size}")
    print()

    X_train_dl, y_train_dl = prepare_dl_data_patches(
        selected_image_names=selected_image_names,
        patch_size=patch_size,
        patches_per_img=patches_per_img
    )

    print()
    print(f"Kształt X_train: {X_train_dl.shape}")
    print(f"Kształt y_train: {y_train_dl.shape}")
    print()

    unet_model = build_simple_unet(
        input_shape=(patch_size, patch_size, 1)
    )

    unet_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=bce_dice_loss,
        metrics=[
            "accuracy",
            dice_coef
        ]
    )

    print(f"Trening sieci U-Net ({epochs} epok, batch_size={batch_size})...")

    history = unet_model.fit(
        X_train_dl,
        y_train_dl,
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    unet_history = history.history

    print()
    print("Sieć U-Net TensorFlow została pomyślnie wytrenowana!")

    unet_model.save(model_path)

    print(f"Sieć U-Net TensorFlow została zapisana do: {model_path}")

    return unet_model


def load_unet(
    model_path=UNET_MODEL_PATH,
    train_if_missing=True,
    **train_kwargs
):
    if os.path.exists(model_path):
        unet_model = models.load_model(
            model_path,
            custom_objects={
                "dice_coef": dice_coef,
                "dice_loss": dice_loss,
                "bce_dice_loss": bce_dice_loss
            }
        )

        print("Sieć U-Net TensorFlow została pomyślnie wczytana")
        return unet_model

    if not train_if_missing:
        raise FileNotFoundError(
            f"Nie znaleziono zapisanego modelu: {model_path}"
        )

    print("Nie znaleziono zapisanego modelu U-Net — uruchamiam trening.")
    return train_unet(
        model_path=model_path,
        **train_kwargs
    )


def predict_dl_full_image(
    img_rgb,
    model,
    threshold=0.01,
    patch_size=128,
    fov_mask=None
):
    h, w, _ = img_rgb.shape

    green = _preprocess_green_channel_clahe(img_rgb)

    if fov_mask is not None:
        fov = fov_mask > 0
        green[~fov] = 0

    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size

    padded_img = np.pad(
        green,
        ((0, pad_h), (0, pad_w)),
        mode="reflect"
    )

    patches = []
    coords = []

    for y in range(0, padded_img.shape[0], patch_size):
        for x in range(0, padded_img.shape[1], patch_size):
            patches.append(padded_img[y:y + patch_size, x:x + patch_size])
            coords.append((y, x))

    patches = np.array(patches, dtype=np.float32)
    patches = np.expand_dims(patches, axis=-1)

    preds = model.predict(
        patches,
        batch_size=8,
        verbose=0
    )

    pred_padded = np.zeros_like(padded_img, dtype=np.float32)

    for i, (y, x) in enumerate(coords):
        pred_padded[y:y + patch_size, x:x + patch_size] = preds[i, :, :, 0]

    raw_pred = pred_padded[:h, :w]

    if fov_mask is not None:
        raw_pred[~fov] = 0

    pred_bin = (raw_pred > threshold).astype(np.uint8)

    if fov_mask is not None:
        pred_bin[~fov] = 0

    return pred_bin


def retinal_vessel_segmentation_unet(
    image,
    mask,
    model,
    threshold=0.01,
    patch_size=128
):
    pred_bin = predict_dl_full_image(
        img_rgb=image,
        model=model,
        threshold=threshold,
        patch_size=patch_size,
        fov_mask=mask
    )

    return pred_bin.astype(np.uint8) * 255


## Training

### Functions

In [7]:
def split_image_names_into_sets(image_names, n_sets=5, random_state=42):
    rng = np.random.default_rng(random_state)
    shuffled_names = np.array(image_names)
    rng.shuffle(shuffled_names)

    image_sets = np.array_split(shuffled_names, n_sets)

    image_sets = [
        list(image_set)
        for image_set in image_sets
    ]

    return image_sets

def format_list_in_columns(items, columns=3):
    rows = []

    for i in range(0, len(items), columns):
        row_items = items[i:i + columns]

        cells = "".join([
            f"<td style='padding: 3px 15px 3px 0; font-family: monospace;'>{item}</td>"
            for item in row_items
        ])

        rows.append(f"<tr>{cells}</tr>")

    return "<table>" + "".join(rows) + "</table>"

### Training Machine Learning Classifier

In [8]:
ml_image_sets = split_image_names_into_sets(
    image_names,
    n_sets=5,
    random_state=42
)

test_set_selector = widgets.Dropdown(
    options=[
        (f"Zbiór testowy {i + 1} ({len(ml_image_sets[i])} obrazów)", i)
        for i in range(len(ml_image_sets))
    ],
    description="Test:"
)


samples_per_image_slider = widgets.IntSlider(
    value=5000,
    min=1000,
    max=15000,
    step=1000,
    description="Ilość próbek na obraz:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)


train_ml_button = widgets.Button(
    description="Trenuj klasyfikator",
    button_style="success",
    icon="cogs"
)


ml_train_output = widgets.Output()

ml_sets_preview = widgets.HTML(
    value="",
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        width="650px"
    )
)


def get_selected_train_test_sets():
    test_set_index = test_set_selector.value

    test_image_names = sorted(ml_image_sets[test_set_index])

    train_image_names = []

    for i, image_set in enumerate(ml_image_sets):
        if i != test_set_index:
            train_image_names.extend(image_set)

    train_image_names = sorted(train_image_names)

    return train_image_names, test_image_names, test_set_index

def show_selected_ml_sets(change=None):
    train_image_names, test_image_names, test_set_index = get_selected_train_test_sets()

    test_html = format_list_in_columns(
        test_image_names,
        columns=3
    )

    train_html = format_list_in_columns(
        train_image_names,
        columns=9
    )

    ml_sets_preview.value = f"""
    <h4>Wybrany zbiór testowy: {test_set_index + 1}</h4>

    <div style="display: flex; gap: 40px; align-items: flex-start;">

        <div>
            <b>Obrazy testowe ({len(test_image_names)}):</b>
            {test_html}
        </div>

        <div>
            <b>Obrazy treningowe ({len(train_image_names)}):</b>
            {train_html}
        </div>

    </div>
    """

def evaluate_classifier(classifier, test_image_names, patch_size=5):
    metrics_per_image = []

    for image_name in test_image_names:
        print(f"Testowanie obrazu: {image_name}")

        raw_image, gs_image, mask = load_images(image_name)

        generated_image_ml = retinal_vessel_segmentation_machine_learning(
            image=raw_image,
            mask=mask,
            classifier=classifier,
            patch_size=patch_size
        )

        metrics = calculate_metrics(
            gs_image=gs_image,
            generated_image=generated_image_ml,
            mask=mask
        )

        metrics_per_image.append((image_name, metrics))

    print()
    print_average_metrics(metrics_per_image)

    return metrics_per_image


def train_ml_classifier_from_selected_set(button):
    global rf_classifier
    global rf_metrics_per_image

    train_ml_button.disabled = True

    with ml_train_output:
        clear_output(wait=True)

        train_image_names, test_image_names, test_set_index = get_selected_train_test_sets()

        print("Rozpoczynam trenowanie klasyfikatora")
        print()

        rf_classifier = train_random_forest_retinal_classifier(
            train_image_names=train_image_names,
            samples_per_image=samples_per_image_slider.value,
            patch_size=5,
            random_state=42
        )

        print("=" * 80)
        print("Ocena zdolności predykcyjnych na obrazach testowych")
        print("=" * 80)
        print()

        rf_metrics_per_image = evaluate_classifier(
            classifier=rf_classifier,
            test_image_names=test_image_names,
            patch_size=5
        )

        print()

    train_ml_button.disabled = False


test_set_selector.observe(show_selected_ml_sets, names="value")
train_ml_button.on_click(train_ml_classifier_from_selected_set)


left_ml_panel = widgets.VBox([
    test_set_selector,
    samples_per_image_slider,
    train_ml_button
])

top_ml_panel = widgets.HBox(
    [
        left_ml_panel,
        ml_sets_preview
    ],
    layout=widgets.Layout(
        width="100%"
    )
)


display(
    widgets.VBox([
        widgets.HTML("<h3>Trenowanie klasyfikatora Random Forest</h3>"),
        top_ml_panel,
        ml_train_output
    ])
)


show_selected_ml_sets()

### Training Deep Learning U-Net


In [9]:
# ---------------- U-NET TRAINING UI ----------------
unet_num_train_slider = widgets.IntSlider(
    value=min(15, len(image_names)),
    min=1,
    max=len(image_names),
    step=1,
    description="Obrazy treningowe:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

unet_patches_per_img_slider = widgets.IntSlider(
    value=150,
    min=10,
    max=500,
    step=10,
    description="Patche / obraz:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

unet_epochs_slider = widgets.IntSlider(
    value=40,
    min=1,
    max=100,
    step=1,
    description="Epoki:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

unet_batch_size_slider = widgets.IntSlider(
    value=16,
    min=1,
    max=64,
    step=1,
    description="Batch size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

unet_learning_rate_slider = widgets.FloatLogSlider(
    value=0.001,
    base=10,
    min=-5,
    max=-2,
    step=0.25,
    readout_format=".5f",
    description="Learning rate:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

unet_threshold_slider = widgets.FloatSlider(
    value=0.01,
    min=0.001,
    max=0.2,
    step=0.001,
    readout_format=".3f",
    description="Threshold do oceny:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

train_unet_button = widgets.Button(
    description="Trenuj i zapisz U-Net",
    button_style="success",
    icon="cogs",
    layout=widgets.Layout(width="180px")
)

load_unet_button = widgets.Button(
    description="Wczytaj / trenuj",
    button_style="info",
    icon="folder-open",
    layout=widgets.Layout(width="160px")
)

unet_train_output = widgets.Output()

unet_sets_preview = widgets.HTML(
    value="",
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        width="700px"
    )
)


PATCH_SIZE_UNET = 128


def get_unet_training_names():
    return image_names[:unet_num_train_slider.value]


def show_unet_training_preview(change=None):
    selected_image_names = get_unet_training_names()

    selected_html = format_list_in_columns(
        selected_image_names,
        columns=6
    )

    total_patches = (
        len(selected_image_names)
        * unet_patches_per_img_slider.value
    )

    model_exists = os.path.exists(UNET_MODEL_PATH)
    model_status = (
        "<span style='color: green; font-weight: 600;'>znaleziony</span>"
        if model_exists
        else "<span style='color: #b36b00; font-weight: 600;'>brak — trzeba wytrenować</span>"
    )

    unet_sets_preview.value = f"""
    <h4>Prosty U-Net patch-based</h4>

    <div>
        <b>Model:</b> 128×128×1, encoder 16→32, bottleneck 64, decoder 32→16<br>
        <b>Preprocessing:</b> CLAHE na kanale zielonym + normalizacja do [0, 1]<br>
        <b>Funkcja straty:</b> BCE + Dice loss<br>
        <b>Plik modelu:</b> <code>{UNET_MODEL_PATH}</code> — {model_status}<br>
        <b>Liczba patchy w treningu:</b> {total_patches}
    </div>

    <br>

    <div>
        <b>Obrazy użyte do treningu ({len(selected_image_names)}):</b>
        {selected_html}
    </div>
    """


def evaluate_unet_on_selected_images(
    model,
    selected_image_names,
    threshold=0.01,
    patch_size=128
):
    metrics_per_image = []

    for image_name in selected_image_names:
        print(f"Szybka ocena obrazu: {image_name}")

        raw_image, gs_image, fov_mask = load_images(image_name)

        generated_image_unet = retinal_vessel_segmentation_unet(
            image=raw_image,
            mask=fov_mask,
            model=model,
            threshold=threshold,
            patch_size=patch_size
        )

        metrics = calculate_metrics(
            gs_image=gs_image,
            generated_image=generated_image_unet,
            mask=fov_mask
        )

        metrics_per_image.append((image_name, metrics))

    print()
    print_average_metrics(metrics_per_image)

    return metrics_per_image


def train_unet_from_ui(button):
    global unet_model
    global unet_metrics_per_image

    train_unet_button.disabled = True
    load_unet_button.disabled = True

    try:
        with unet_train_output:
            clear_output(wait=True)

            selected_image_names = get_unet_training_names()

            print("Rozpoczynam trening prostego U-Net")
            print("=" * 80)
            print()

            unet_model = train_unet(
                selected_image_names=selected_image_names,
                num_train=len(selected_image_names),
                patch_size=PATCH_SIZE_UNET,
                patches_per_img=unet_patches_per_img_slider.value,
                epochs=unet_epochs_slider.value,
                batch_size=unet_batch_size_slider.value,
                learning_rate=unet_learning_rate_slider.value,
                model_path=UNET_MODEL_PATH,
                random_state=42
            )

            print()
            print("=" * 80)
            print("Szybka ocena na 3 obrazach treningowych")
            print("=" * 80)
            print()

            unet_metrics_per_image = evaluate_unet_on_selected_images(
                model=unet_model,
                selected_image_names=selected_image_names[:3],
                threshold=unet_threshold_slider.value,
                patch_size=PATCH_SIZE_UNET
            )

    finally:
        train_unet_button.disabled = False
        load_unet_button.disabled = False
        show_unet_training_preview()


def load_unet_from_ui(button):
    global unet_model

    train_unet_button.disabled = True
    load_unet_button.disabled = True

    try:
        with unet_train_output:
            clear_output(wait=True)

            selected_image_names = get_unet_training_names()

            unet_model = load_unet(
                model_path=UNET_MODEL_PATH,
                train_if_missing=True,
                selected_image_names=selected_image_names,
                num_train=len(selected_image_names),
                patch_size=PATCH_SIZE_UNET,
                patches_per_img=unet_patches_per_img_slider.value,
                epochs=unet_epochs_slider.value,
                batch_size=unet_batch_size_slider.value,
                learning_rate=unet_learning_rate_slider.value,
                random_state=42
            )

            print()
            print("Model jest gotowy do użycia w panelu segmentacji.")

    finally:
        train_unet_button.disabled = False
        load_unet_button.disabled = False
        show_unet_training_preview()


unet_num_train_slider.observe(show_unet_training_preview, names="value")
unet_patches_per_img_slider.observe(show_unet_training_preview, names="value")
unet_epochs_slider.observe(show_unet_training_preview, names="value")
unet_batch_size_slider.observe(show_unet_training_preview, names="value")
unet_learning_rate_slider.observe(show_unet_training_preview, names="value")
unet_threshold_slider.observe(show_unet_training_preview, names="value")

train_unet_button.on_click(train_unet_from_ui)
load_unet_button.on_click(load_unet_from_ui)


left_unet_panel = widgets.VBox([
    widgets.HTML("<h4>Parametry treningu</h4>"),
    unet_num_train_slider,
    unet_patches_per_img_slider,
    unet_epochs_slider,
    unet_batch_size_slider,
    unet_learning_rate_slider,
    unet_threshold_slider,
    widgets.HBox([
        train_unet_button,
        load_unet_button
    ])
])


top_unet_panel = widgets.HBox(
    [
        left_unet_panel,
        unet_sets_preview
    ],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start"
    )
)


display(
    widgets.VBox([
        widgets.HTML("<h3>Trenowanie prostego U-Net</h3>"),
        top_unet_panel,
        unet_train_output
    ])
)


show_unet_training_preview()


## App


In [10]:
# ---------------- UI ----------------
image_selector = widgets.Dropdown(
    options=image_names,
    description="Obraz:"
)

segment_button = widgets.Button(
    description="Segmentacja",
    button_style="success",
    icon="play"
)

view_mode = widgets.ToggleButtons(
    options=[
        ("Maski", "masks"),
        ("Nałożenia", "overlays")
    ],
    description="Widok:"
)

unet_threshold_slider_segmentation = widgets.FloatSlider(
    value=0.01,
    min=0.001,
    max=0.2,
    step=0.001,
    readout_format=".3f",
    description="U-Net threshold:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

preview_output = widgets.Output()
result_output = widgets.Output()
metrics_output = widgets.Output()

last_result = {}

PATCH_SIZE_UNET = 128

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Postęp:",
    bar_style="",
    layout=widgets.Layout(width="250px", visibility="hidden")
)

progress_label = widgets.HTML(
    value="",
    layout=widgets.Layout(visibility="hidden")
)


# ---------------- CLEANING ----------------
def on_image_change(change):
    global last_result
    last_result = {}

    with preview_output:
        clear_output(wait=True)
    with result_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)

    show_preview()


# ---------------- PREVIEW ----------------
def show_preview(change=None):
    with preview_output:
        clear_output(wait=True)

        image_name = image_selector.value
        raw_image, _, _ = load_images(image_name)

        plt.figure(figsize=(5, 5))
        plt.imshow(raw_image)
        plt.title(image_name)
        plt.axis("off")
        plt.show()


# ---------------- RESULT RENDERING ----------------
def render_result():
    if not last_result:
        return

    raw_image = last_result["raw_image"]
    gs_image = last_result["gs_image"]

    generated_image_ip = last_result["generated_image_ip"]
    generated_image_rf = last_result["generated_image_rf"]
    generated_image_unet = last_result["generated_image_unet"]

    gs_overlay = last_result["gs_overlay"]
    generated_overlay_ip = last_result["generated_overlay_ip"]
    generated_overlay_rf = last_result["generated_overlay_rf"]
    generated_overlay_unet = last_result["generated_overlay_unet"]

    with result_output:
        clear_output(wait=True)

        fig, axes = plt.subplots(1, 5, figsize=(25, 5))

        if view_mode.value == "masks":
            axes[0].imshow(raw_image)
            axes[0].set_title("Obraz wejściowy")

            axes[1].imshow(gs_image, cmap="gray")
            axes[1].set_title("Maska ekspercka")

            axes[2].imshow(generated_image_ip, cmap="gray")
            axes[2].set_title("Wygenerowana maska\nImage Processing")

            axes[3].imshow(generated_image_rf, cmap="gray")
            axes[3].set_title("Wygenerowana maska\nRandom Forest")

            axes[4].imshow(generated_image_unet, cmap="gray")
            axes[4].set_title("Wygenerowana maska\nU-Net simple patch-based")

        elif view_mode.value == "overlays":
            axes[0].imshow(raw_image)
            axes[0].set_title("Obraz wejściowy")

            axes[1].imshow(gs_overlay)
            axes[1].set_title("Nałożona maska ekspercka")

            axes[2].imshow(generated_overlay_ip)
            axes[2].set_title("Nałożona maska\nImage Processing")

            axes[3].imshow(generated_overlay_rf)
            axes[3].set_title("Nałożona maska\nRandom Forest")

            axes[4].imshow(generated_overlay_unet)
            axes[4].set_title("Nałożona maska\nU-Net simple patch-based")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()
        plt.show()


# ---------------- METRICS PRINTING ----------------
def print_metrics(name, metrics):
    print(name)
    print("-" * len(name))

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k:12s}: {v:.4f}")
        else:
            print(f"{k:12s}: {v}")

    print()


# ---------------- SEGMENTATION ----------------
def run_segmentation(button):
    global last_result

    segment_button.disabled = True

    progress_bar.layout.visibility = "visible"
    progress_label.layout.visibility = "visible"

    progress_bar.value = 0
    progress_bar.bar_style = ""
    progress_label.value = "Przygotowywanie danych..."

    try:
        if "rf_classifier" not in globals():
            raise ValueError(
                "Najpierw wytrenuj klasyfikator Random Forest. "
                "Model powinien być zapisany w zmiennej rf_classifier."
            )

        if "unet_model" not in globals():
            raise ValueError(
                "Najpierw wytrenuj sieć U-Net. "
                "Model powinien być zapisany w zmiennej unet_model."
            )

        image_name = image_selector.value

        progress_bar.value = 10
        progress_label.value = "Wczytywanie obrazu..."

        raw_image, gs_image, mask = load_images(image_name)

        progress_bar.value = 20
        progress_label.value = "Segmentacja metodą przetwarzania obrazu..."

        generated_image_ip = retinal_vessel_segmentation_image_processing(
            raw_image,
            mask
        )

        progress_bar.value = 45
        progress_label.value = "Segmentacja metodą Random Forest..."

        generated_image_rf = retinal_vessel_segmentation_machine_learning(
            image=raw_image,
            mask=mask,
            classifier=rf_classifier,
            patch_size=5
        )

        if generated_image_rf.max() == 1:
            generated_image_rf = generated_image_rf * 255

        progress_bar.value = 65
        progress_label.value = "Segmentacja metodą U-Net simple patch-based..."

        generated_image_unet = retinal_vessel_segmentation_unet(
            image=raw_image,
            mask=mask,
            model=unet_model,
            threshold=unet_threshold_slider_segmentation.value,
            patch_size=PATCH_SIZE_UNET
        )

        if generated_image_unet.max() == 1:
            generated_image_unet = generated_image_unet * 255

        progress_bar.value = 80
        progress_label.value = "Generowanie podglądu..."

        gs_overlay = generate_overlay(raw_image, gs_image)
        generated_overlay_ip = generate_overlay(raw_image, generated_image_ip)
        generated_overlay_rf = generate_overlay(raw_image, generated_image_rf)
        generated_overlay_unet = generate_overlay(raw_image, generated_image_unet)

        last_result = {
            "raw_image": raw_image,
            "gs_image": gs_image,
            "generated_image_ip": generated_image_ip,
            "generated_image_rf": generated_image_rf,
            "generated_image_unet": generated_image_unet,
            "gs_overlay": gs_overlay,
            "generated_overlay_ip": generated_overlay_ip,
            "generated_overlay_rf": generated_overlay_rf,
            "generated_overlay_unet": generated_overlay_unet
        }

        render_result()

        progress_bar.value = 90
        progress_label.value = "Obliczanie metryk..."

        metrics_ip = calculate_metrics(
            gs_image=gs_image,
            generated_image=generated_image_ip,
            mask=mask
        )

        metrics_rf = calculate_metrics(
            gs_image=gs_image,
            generated_image=generated_image_rf,
            mask=mask
        )

        metrics_unet = calculate_metrics(
            gs_image=gs_image,
            generated_image=generated_image_unet,
            mask=mask
        )

        with metrics_output:
            clear_output(wait=True)

            print("Metryki:")
            print()
            print(f"U-Net patch size: {PATCH_SIZE_UNET}")
            print(f"U-Net threshold: {unet_threshold_slider_segmentation.value:.3f}")
            print()

            print_metrics(
                "Image Processing",
                metrics_ip
            )

            print_metrics(
                "Random Forest",
                metrics_rf
            )

            print_metrics(
                "U-Net simple patch-based",
                metrics_unet
            )

        progress_bar.value = 100
        progress_bar.bar_style = "success"
        progress_label.value = "Segmentacja zakończona."

    except Exception as e:
        progress_bar.bar_style = "danger"
        progress_label.value = f"Błąd segmentacji: {e}"

    finally:
        segment_button.disabled = False


# ---------------- VIEW MODE CHANGE ----------------
def on_view_mode_change(change):
    render_result()


# ---------------- OBSERVERS ----------------
image_selector.observe(on_image_change, names="value")
view_mode.observe(on_view_mode_change, names="value")
segment_button.on_click(run_segmentation)


# ---------------- LAYOUT ----------------
left_panel = widgets.VBox([
    image_selector,
    view_mode,
    unet_threshold_slider_segmentation,
    segment_button,
    progress_bar,
    progress_label
])

top_panel = widgets.HBox([
    left_panel,
    preview_output
])

display(
    widgets.VBox([
        top_panel,
        result_output,
        metrics_output
    ])
)

# pierwszy podgląd
show_preview()